In [ ]:
import numpy as np
import pandas as pd
import image_dataset_loader
import cv2
import torch
import torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from PIL import Image
import os, sys

path = "C:/Users/surya/OneDrive/Desktop/Images/AIDER/train/traffic_incident_image/"
dirs = os.listdir( path )

def resize():
    for item in dirs:
        if os.path.isfile(path+item):
            im = Image.open(path+item)
            f, e = os.path.splitext(path)
            i, j = os.path.splitext(item)
            imResize = im.resize((224,224), Image.Resampling.LANCZOS)
            imResize.save(f+i+'.jpg', 'JPEG', quality=90)

resize()

In [ ]:
(x_train, y_train), (x_test, y_test) = image_dataset_loader.load("C:/Users/surya/OneDrive/Desktop/Images/AIDER/", ['train', 'test'])

In [ ]:
labels = image_dataset_loader._sorted_class_names("C:/Users/surya/OneDrive/Desktop/Images/AIDER/train/")
print(labels)

In [ ]:
print(x_test.shape)

In [ ]:
print(y_test.shape)

In [ ]:
x_test = x_test/255
x_train = x_train/255

In [ ]:
print(y_test[0:25])

In [ ]:
class Disaster(nn.Module):
    def __init__(self):
        super(Disaster,self).__init__()
        self.l1 = nn.Conv2d(3,64,(3,3),1,1)
        self.relu = nn.ReLU()
        self.l2 = nn.Conv2d(64,32,(3,3),1,1)
        self.pool = nn.MaxPool2d(kernel_size=(8,8))
        self.flat = nn.Flatten()
        self.l3 = nn.Linear(25088,1000)
        self.l4 = nn.Linear(1000,5)
        self.drop1 = nn.Dropout(0.2)
        self.drop2 = nn.Dropout(0.3)
        
    def forward(self,x):
        #input = 3*224*224, output=64*224*224
        x = self.relu(self.l1(x))
        x = self.drop1(x)
        #input = 64*224*224, output = 32*224*224
        x = self.relu(self.l2(x))
        #input = 32*224*224, output = 32*28*28
        x = self.pool(x)
        #input = 32*28*28, output = 25088
        x = self.flat(x)
        #input = 25088, hidden = 1000, output = 5
        x = self.drop2(self.l3(x))
        x = self.l4(x)
        
        return x
        
        

In [ ]:
import torch.nn as nn
import torch.optim as optim

x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train.astype(np.float32), dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test.astype(np.float32), dtype=torch.long)
trainset = torch.utils.data.TensorDataset(x_train, y_train)
testset = torch.utils.data.TensorDataset(x_test, y_test)


batch_size = 5
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
testloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)

In [ ]:
for im, l in trainloader:
    print(im.shape)
    print(l.shape)

In [ ]:
model = Disaster()
train_acc = []
train_loss = []
test_acc = []
test_loss = []
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())
n_epochs = 20
for epoch in range(n_epochs):
    acc = 0
    count = 0
    running_loss = 0
    for inputs, labels in trainloader:
        # forward, backward, and then weight update
        inputs = torch.rot90(inputs,1,[1,3])
        model = model.to(torch.device('cuda'))
        inputs = inputs.to(torch.device('cuda'))
        labels = labels.to(torch.device('cuda'))
        y_pred = model(inputs)
        loss = loss_fn(y_pred, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        acc += (torch.argmax(y_pred, 1) == labels).float().sum()
        count += len(labels)
    train_acc.append(100*acc/count)
    train_loss.append(running_loss/len(trainloader))
    print("Epoch %d: model accuracy %.2f%%, losses %.4f" % (epoch, 100*acc/count,(running_loss/len(trainloader))))
 


In [ ]:
#testing loop
test_losses = []
test_accu= []
n_epochs = 20
for epoch in range(n_epochs):
    running_loss = 0.0
    total_samples = 0
    correct_predictions = 0
    for inputs, labels in trainloader:
        # forward, backward, and then weight update
        inputs = torch.rot90(inputs,1,[1,3])
        model = model.to(torch.device('cuda'))
        inputs = inputs.to(torch.device('cuda'))
        labels = labels.to(torch.device('cuda'))
        y_pred = model(inputs)
        loss = loss_fn(y_pred, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        predicted = torch.argmax(y_pred)
        correct_predictions += (predicted == labels.squeeze().long()).sum().item()
        total_samples += len(labels)
    # Print the average loss for each epoch
    accuracy = 100*correct_predictions / total_samples
    test_accu.append(accuracy)
    print(accuracy)
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {running_loss / len(testloader):.4f}")
    test_losses.append(running_loss/len(testloader))

In [ ]:
import pickle

Pkl_Filename = "Disaster_model.pkl"  

with open(Pkl_Filename, 'wb') as file:  
    pickle.dump(model, file)

with open(Pkl_Filename, 'rb') as file:  
    Model = pickle.load(file)

Model

In [ ]:
import PIL
from PIL import Image
import os,sys
import cv2 
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
im = Image.open('C:/Users/surya/OneDrive/Desktop/Images/Trial.jpg')
reIM = im.resize((224,224), Image.Resampling.LANCZOS)
imgplot = plt.imshow(reIM)
img = cv2.imread('C:/Users/surya/OneDrive/Desktop/Images/Trial.jpg')
print(img.shape)
res = cv2.resize(img, dsize=(224, 224), interpolation=cv2.INTER_CUBIC)
res = res/255
print(res.shape)


In [ ]:
print(res)

In [ ]:
y_pred = Model(res)
print(y_pred)